# Classificação do Dataset Adult com Agglomerative Clustering Supervisionado

Este notebook implementa a classificação do dataset Adult utilizando Agglomerative Clustering supervisionado, com download automático dos dados, avaliação em 30 execuções e salvamento dos resultados (matriz de confusão e gráficos) na pasta `img`.

In [1]:
# Importar bibliotecas necessárias
import os
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Carregar e visualizar o dataset Adult a partir da pasta data
import pandas as pd

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income'
]
data_path = 'data/adult.data'
test_path = 'data/adult.test'
df_train = pd.read_csv(data_path, names=columns, sep=',', skipinitialspace=True)
df_test = pd.read_csv(test_path, names=columns, sep=',', skipinitialspace=True, skiprows=1)
df = pd.concat([df_train, df_test], ignore_index=True)
df = df.replace('?', pd.NA).dropna()
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
# Pré-processamento dos dados
from sklearn.preprocessing import LabelEncoder

def preprocess_adult(df):
    X = df.drop('income', axis=1)
    y = df['income'].apply(lambda x: 1 if '>50K' in x else 0)
    X = pd.get_dummies(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y.values

X, y = preprocess_adult(df)
print(f"Shape dos dados após preprocessamento: {X.shape}")

Shape dos dados após preprocessamento: (45222, 104)


In [4]:
# Implementação do Agglomerative Clustering supervisionado
from sklearn.metrics import pairwise_distances

def elbow_method_agglomerative(X, max_k=15):
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import pdist
    dists = []
    for k in range(2, max_k+1):
        Z = linkage(X, method='ward')
        labels = fcluster(Z, k, criterion='maxclust')
        # Soma das distâncias intra-cluster
        dist = 0
        for i in range(1, k+1):
            cluster_points = X[labels == i]
            if len(cluster_points) > 1:
                dist += np.sum(pairwise_distances(cluster_points, [cluster_points.mean(axis=0)]))
        dists.append(dist)
    deltas = np.diff(dists)
    elbow = np.argmin(deltas) + 2
    return range(2, max_k+1)[elbow]

class SupervisedAgglomerativeClassifier:
    def __init__(self, max_k=15):
        self.max_k = max_k
        self.n_clusters = None
        self.model = None
        self.cluster_labels = None
    def fit(self, X, y):
        self.n_clusters = elbow_method_agglomerative(X, self.max_k)
        self.model = AgglomerativeClustering(n_clusters=self.n_clusters, linkage='ward')
        clusters = self.model.fit_predict(X)
        self.cluster_labels = np.zeros(self.n_clusters, dtype=int)
        for i in range(self.n_clusters):
            mask = (clusters == i)
            if np.any(mask):
                self.cluster_labels[i] = mode(y[mask], keepdims=True)[0][0]
            else:
                self.cluster_labels[i] = 0
    def predict(self, X):
        # Aproximação: atribui ao cluster mais próximo do centroide
        from sklearn.neighbors import NearestCentroid
        centroids = []
        clusters = self.model.fit_predict(X)
        for i in range(self.n_clusters):
            centroids.append(X[clusters == i].mean(axis=0))
        centroids = np.array(centroids)
        nc = NearestCentroid()
        nc.centroids_ = centroids
        return self.cluster_labels[nc.predict(X)]

In [5]:
# Treinamento, avaliação e salvamento dos resultados
accs = []
cms = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=seed, stratify=y)
    clf = SupervisedAgglomerativeClassifier()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    accs.append(acc)
    cms.append(cm)
    print(f"Execução {seed:02d}: Acurácia = {acc:.4f}")
accs = np.array(accs)
cms = np.array(cms)

# Salvar matrizes e gráficos
np.save('Trabalho03/img/agglomerative_adult_matrizes_confusao.npy', cms)
np.save('Trabalho03/img/agglomerative_adult_acuracias.npy', accs)

# Gráfico de acurácia
plt.figure(figsize=(8,4))
plt.plot(range(1,31), accs, marker='o')
plt.title('Acurácia em cada execução (Agglomerative + Adult)')
plt.xlabel('Execução')
plt.ylabel('Acurácia')
plt.grid()
plt.savefig('Trabalho03/img/agglomerative_adult_acuracia.png')
plt.show()

# Matriz de confusão média
cm_mean = np.round(cms.mean(axis=0)).astype(int)
sns.heatmap(cm_mean, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão Média (Agglomerative + Adult)')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.savefig('Trabalho03/img/agglomerative_adult_cm_media.png')
plt.show()

print(f"Acurácia média: {accs.mean():.4f}")
print(f"Desvio padrão da acurácia: {accs.std():.4f}")

KeyboardInterrupt: 